# Partie 3 — Évaluation, interprétation et choix du modèle final

Cette partie s’appuie sur le modèle logit ordonné estimé précédemment.  
L’objectif est d’évaluer la qualité du modèle, d’interpréter économiquement les résultats obtenus et de justifier le choix du modèle final.

La variable dépendante est `score_yuka_ordonne`, une variable qualitative ordonnée allant de 1 à 4 :

- 1 = Mauvais
- 2 = Médiocre
- 3 = Bon
- 4 = Excellent

In [1]:
import pandas as pd
import numpy as np

from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.metrics import confusion_matrix, accuracy_score

# Chargement de la base clean

In [2]:
df = pd.read_csv("data/jeu_donnees_yuka.csv", sep=";")

df.head()

,id_produit,calories_100g,sucres_100g,graisses_saturees_100g,sel_100g,fibres_100g,proteines_100g,nb_additifs,bio,ultra_transforme,score_yuka_ordonne,classe_yuka
0,1,229.8,10.3,3.7,1.22,2.7,0.0,4,1,1,1,Mauvais
1,2,282.5,23.6,6.0,0.87,1.2,1.9,0,0,0,1,Mauvais
2,3,446.7,32.8,3.3,1.03,7.0,12.5,2,0,0,1,Mauvais
3,4,121.8,20.8,6.0,0.07,5.1,2.0,2,1,1,2,Médiocre
4,5,257.0,16.6,3.3,1.64,1.4,10.6,0,0,0,2,Médiocre


# Modèle de Havar

In [3]:
y = df["score_yuka_ordonne"]

X = df[
    [
        "calories_100g",
        "sucres_100g",
        "graisses_saturees_100g",
        "sel_100g",
        "fibres_100g",
        "proteines_100g",
        "nb_additifs",
        "bio",
        "ultra_transforme",
    ]
]

model = OrderedModel(y, X, distr="logit")
result = model.fit(method="bfgs")

result.summary()

Optimization terminated successfully.
         Current function value: 0.165717
         Iterations: 67
         Function evaluations: 73
         Gradient evaluations: 73


<class 'statsmodels.iolib.summary.Summary'>
"""
                             OrderedModel Results                             
==============================================================================
Dep. Variable:     score_yuka_ordonne   Log-Likelihood:                -165.72
Model:                   OrderedModel   AIC:                             355.4
Method:            Maximum Likelihood   BIC:                             414.3
Date:                Fri, 01 May 2026                                         
Time:                        11:50:57                                         
No. Observations:                1000                                         
Df Residuals:                     988                                         
Df Model:                           9                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
calories_100g             -0.0774      0.006    -12.898      0.000      -0.089      -0.066
sucres_100g               -0.7306      0.058    -12.679      0.000      -0.844      -0.618
graisses_saturees_100g    -2.0544      0.163    -12.615      0.000      -2.374      -1.735
sel_100g                  -9.2493      0.751    -12.324      0.000     -10.720      -7.778
fibres_100g                2.0615      0.169     12.191      0.000       1.730       2.393
proteines_100g             0.9454      0.077     12.223      0.000       0.794       1.097
nb_additifs               -2.9282      0.238    -12.328      0.000      -3.394      -2.463
bio                        7.6323      0.679     11.246      0.000       6.302       8.963
ultra_transforme          -7.6004      0.658    -11.559      0.000      -8.889      -6.312
1/2                      -42.1381      3.244    -12.989      0.000     -48.496     -35.780
2/3                        2.5219      0.080     31.450      0.000       2.365       2.679
3/4                        2.5223      0.079     31.979      0.000       2.368       2.677
==========================================================================================
"""

# 6. Évaluation du modèle

L’évaluation du modèle permet de vérifier si le modèle logit ordonné estimé est statistiquement satisfaisant et économiquement pertinent.  
Nous analysons d’abord la significativité des coefficients, puis les critères de qualité du modèle.

# 6.1 Test indiv

In [4]:
resultats = pd.DataFrame({
    "Coefficient": result.params,
    "P-value": result.pvalues
})

resultats

,Coefficient,P-value
calories_100g,-0.077425,4.647836e-38
sucres_100g,-0.730577,7.770420e-37
graisses_saturees_100g,-2.054391,1.738137e-36
sel_100g,-9.249304,6.753339e-35
fibres_100g,2.061492,3.470404e-34
proteines_100g,0.945445,2.328426e-34
nb_additifs,-2.928151,6.376629e-35
bio,7.632294,2.434784e-29
ultra_transforme,-7.600353,6.653549e-31
1/2,-42.138080,1.404181e-38




Le tableau des coefficients montre que l’ensemble des variables explicatives du modèle sont statistiquement significatives au seuil de 5 %, puisque leurs p-values sont toutes inférieures à 0,05.

Les coefficients négatifs des calories, des sucres, des graisses saturées, du sel, du nombre d’additifs et du caractère ultra-transformé indiquent que ces variables réduisent la probabilité d’obtenir une meilleure note Yuka.

À l’inverse, les coefficients positifs des fibres, des protéines et du caractère biologique du produit indiquent qu’une hausse de ces variables est associée à une probabilité plus élevée d’obtenir une bonne ou excellente note.

Ces résultats sont cohérents avec la logique nutritionnelle de Yuka : les éléments défavorables à la qualité nutritionnelle sont pénalisés, tandis que les éléments considérés comme plus favorables sont valorisés.

# Test LR

In [6]:
import scipy.stats as stats

# Log-vraisemblance du modèle estimé
ll_full = result.llf

# Approximation du modèle nul
# (log-vraisemblance d'un modèle uniforme)
n = len(y)
ll_null = -n * np.log(len(np.unique(y)))

# Statistique du test LR
LR_stat = 2 * (ll_full - ll_null)

# Degrés de liberté
df = X.shape[1]

# p-value
p_value_lr = stats.chi2.sf(LR_stat, df)

print("Statistique LR :", LR_stat)
print("p-value :", p_value_lr)

Statistique LR : 2441.1541727980325
p-value : 0.0


La p-value du test du ratio de vraisemblance est extrêmement faible (proche de 0), ce qui conduit à rejeter très fortement l’hypothèse nulle selon laquelle les coefficients sont conjointement nuls.

Ainsi, le modèle est globalement très significatif : les variables explicatives contribuent fortement à expliquer la notation Yuka.